# Imports and Functions

In [22]:
from collections import defaultdict
import json
import os
import glob
from typing import List, Callable, Union

import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, mean_absolute_error, root_mean_squared_error,
                             precision_recall_curve, r2_score, roc_auc_score, mean_absolute_percentage_error, auc)
%load_ext autoreload
%autoreload 2

from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from chemprop import models

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
def parse_indices(idxs):
    """Parses a string of indices into a list of integers. e.g. '0,1,2-4' -> [0, 1, 2, 3, 4]"""
    if isinstance(idxs, str):
        indices = []
        for idx in idxs.split(","):
            if "-" in idx:
                start, end = map(int, idx.split("-"))
                indices.extend(range(start, end + 1))
            else:
                indices.append(int(idx))
        return indices
    return idxs

def prc_auc(targets: List[int], preds: List[float]) -> float:
    """
    Computes the area under the precision-recall curve.

    :param targets: A list of binary targets.
    :param preds: A list of prediction probabilities.
    :return: The computed prc-auc.
    """
    precision, recall, _ = precision_recall_curve(targets, preds)
    return auc(recall, precision)

def get_metric_func(metric: str):
    r"""
    Gets the metric function corresponding to a given metric name.

    Supports:

    * :code:`roc-auc`: Area under the receiver operating characteristic curve
    * :code:`prc-auc`: Area under the precision recall curve
    * :code:`ap`: Average precision from prediction scores
    * :code:`rmse`: Root mean squared error
    * :code:`mae`: Mean absolute error
    * :code:`r2`: Coefficient of determination R\ :superscript:`2`

    :param metric: Metric name.
    :return: A metric function which takes as arguments a list of targets and a list of predictions and returns.
    """
    if metric == 'roc-auc':
        return roc_auc_score

    if metric == 'prc-auc':
        return prc_auc
    
    if metric == 'ap':
        return average_precision_score

    if metric == 'rmse':
        return root_mean_squared_error
    
    if metric == 'mae':
        return mean_absolute_error

    if metric == 'r2':
        return r2_score
    
    raise ValueError(f'Metric "{metric}" not supported.')
    

In [24]:
def evaluate_results(data_path, splits_path, result_dir, num_tasks, metrics, target_columns=None):
    df = pd.read_csv(data_path)
    with open(splits_path, "rb") as json_file:
        split_idxss = json.load(json_file)
    test_indices = [parse_indices(d["test"]) for d in split_idxss]
    test_df = df.iloc[test_indices[0]]
    target_columns=test_df.keys()[-num_tasks:].tolist() if target_columns is None else target_columns

    df_pred_list = []
    files = glob.glob(os.path.join(result_dir, '**', "test_predictions.csv"), recursive=True)
    assert len(files) == 5, f"There should be 5 files; {len(files)} found"
    for file in files:
        df_pred = pd.read_csv(file)[target_columns]
        df_pred_list.append(df_pred)
    df_pred = pd.concat(df_pred_list).groupby(level=0).mean()

    metric_to_func = {metric: get_metric_func(metric) for metric in metrics}

    results = defaultdict(list)
    for column in target_columns:
        for metric, metric_func in metric_to_func.items():
            preds = df_pred[column].tolist()
            targets = test_df[column].tolist()
            results[metric].append(metric_func(targets, preds))
    results = dict(results)

    results_df = pd.DataFrame(results, index=target_columns)
    return results_df

In [25]:
def evaluate_results_multi_fold(data_path, splits_path, result_dir, num_tasks, metrics, target_columns=None, num_folds=5):
    """
    Evaluate results across multiple folds with multiple models per fold.
    
    Args:
        data_path: Path to the dataset CSV file
        splits_path: Path to the multiple_splits.json file
        result_dir: Directory containing fold_X/model_Y/test_predictions.csv structure
        num_tasks: Number of target tasks
        metrics: List of metrics to compute
        target_columns: List of target column names (optional)
        num_folds: Number of folds (default: 5)
    
    Returns:
        dict: Results with mean, std, and fold-wise metrics
    """
    df = pd.read_csv(data_path)
    with open(splits_path, "rb") as json_file:
        split_idxss = json.load(json_file)
    
    # Use the first fold's test set (all folds should have the same test set)
    test_indices = parse_indices(split_idxss[0]["test"])
    test_df = df.iloc[test_indices]
    target_columns = test_df.keys()[-num_tasks:].tolist() if target_columns is None else target_columns
    
    metric_to_func = {metric: get_metric_func(metric) for metric in metrics}
    
    # Store results for each fold
    fold_results = defaultdict(list)
    
    for fold_idx in range(num_folds):
        fold_dir = os.path.join(result_dir, f"fold_{fold_idx}")
        
        # Automatically detect the number of models in this fold
        if not os.path.exists(fold_dir):
            print(f"Warning: Fold directory {fold_dir} does not exist")
            continue
            
        # Find all model directories in this fold
        model_dirs = [d for d in os.listdir(fold_dir) 
                     if os.path.isdir(os.path.join(fold_dir, d)) and d.startswith('model_')]
        
        if not model_dirs:
            print(f"Warning: No model directories found in fold {fold_idx}")
            continue
        
        # Sort model directories by model index
        model_dirs.sort(key=lambda x: int(x.split('_')[1]))
        
        #print(f"Fold {fold_idx}: Found {len(model_dirs)} models: {model_dirs}")
        
        # Collect predictions from all models in this fold
        df_pred_list = []
        for model_dir in model_dirs:
            model_file = os.path.join(fold_dir, model_dir, "test_predictions.csv")
            if os.path.exists(model_file):
                df_pred = pd.read_csv(model_file)[target_columns]
                df_pred_list.append(df_pred)
            else:
                print(f"Warning: Prediction file not found: {model_file}")
        
        if not df_pred_list:
            print(f"Warning: No predictions found for fold {fold_idx}")
            continue
            
        # Average predictions across models for this fold
        df_pred_fold = pd.concat(df_pred_list).groupby(level=0).mean()
        
        # Calculate metrics for this fold
        fold_metrics = defaultdict(list)
        for column in target_columns:
            for metric, metric_func in metric_to_func.items():
                preds = df_pred_fold[column].tolist()
                targets = test_df[column].tolist()
                try:
                    metric_value = metric_func(targets, preds)
                    fold_metrics[metric].append(metric_value)
                except Exception as e:
                    print(f"Error calculating {metric} for fold {fold_idx}, column {column}: {e}")
                    fold_metrics[metric].append(np.nan)
        
        # Store fold results
        for metric in metrics:
            fold_results[metric].append(fold_metrics[metric])
    
    # Calculate summary statistics
    summary_results = {}
    for metric in metrics:
        fold_metric_values = np.array(fold_results[metric])  # Shape: (num_folds, num_targets)
        
        summary_results[f"{metric}_mean"] = np.nanmean(fold_metric_values, axis=0)
        summary_results[f"{metric}_std"] = np.nanstd(fold_metric_values, axis=0, ddof=1)
        summary_results[f"{metric}_folds"] = fold_metric_values
    
    # Create summary DataFrame
    summary_df = pd.DataFrame({
        metric + "_mean": summary_results[f"{metric}_mean"] for metric in metrics
    }, index=target_columns)
    
    # Add std columns
    for metric in metrics:
        summary_df[f"{metric}_std"] = summary_results[f"{metric}_std"]
    
    return {
        'summary': summary_df,
        'fold_results': dict(fold_results),
        'raw_results': summary_results
    }

In [26]:
def compare_methods_statistical(method1_results, method2_results, metrics, method1_name="Method1", method2_name="Method2", alpha=0.05):
    """
    Compare two methods using statistical tests (paired t-test).
    
    Args:
        method1_results: Results from evaluate_results_multi_fold for method 1
        method2_results: Results from evaluate_results_multi_fold for method 2
        metrics: List of metrics to compare
        method1_name: Name of method 1 for display
        method2_name: Name of method 2 for display
    
    Returns:
        pd.DataFrame: Comparison results with p-values
    """
    comparison_results = []
    
    for metric in metrics:
        
        method1_folds = method1_results['raw_results'][f"{metric}_folds"]
        method2_folds = method2_results['raw_results'][f"{metric}_folds"]
        
        # For each target column
        for target_idx in range(method1_folds.shape[1]):
            method1_values = method1_folds[:, target_idx]
            method2_values = method2_folds[:, target_idx]
            
            # Check for NaN values
            if np.any(np.isnan(method1_values)) or np.any(np.isnan(method2_values)):
                print(f"  Warning: NaN values found for {metric}, target {target_idx}")
                t_stat, p_value = np.nan, np.nan
            elif len(method1_values) < 2 or len(method2_values) < 2:
                print(f"  Warning: Not enough samples for {metric}, target {target_idx}")
                t_stat, p_value = np.nan, np.nan
            elif np.allclose(method1_values, method2_values):
                print(f"  Warning: Identical values for {metric}, target {target_idx}")
                t_stat, p_value = 0.0, 1.0
            else:
                try:
                    # Perform paired t-test
                    t_stat, p_value = ttest_rel(method1_values, method2_values)
                except Exception as e:
                    print(f"  Error in t-test for {metric}, target {target_idx}: {e}")
                    t_stat, p_value = np.nan, np.nan
            
            # Calculate means and effect size
            method1_mean = np.nanmean(method1_values)
            method2_mean = np.nanmean(method2_values)
            method1_std = np.nanstd(method1_values, ddof=1)
            method2_std = np.nanstd(method2_values, ddof=1)
            
            # Cohen's d for effect size (equal sample sizes)
            pooled_std = np.sqrt((method1_std**2 + method2_std**2) / 2)
            cohens_d = (method1_mean - method2_mean) / pooled_std if pooled_std > 0 and not np.isnan(pooled_std) else 0

            # thresholds
            stat_significant = p_value < alpha/2 if not np.isnan(p_value) else False
            practical_significant = abs(cohens_d) >= 0.5   # medium effect size as threshold (Cohen’s convention)

            comparison_results.append({
                'metric': metric,
                'target_idx': target_idx,
                f'{method1_name}_mean': method1_mean,
                f'{method1_name}_std': method1_std,
                f'{method2_name}_mean': method2_mean,
                f'{method2_name}_std': method2_std,
                't_statistic': t_stat,
                'p_value': p_value,
                'cohens_d': cohens_d,
                'statistical_significant': stat_significant,
                'practical_significant': practical_significant,
                'overall_significant': stat_significant and practical_significant
            })
    
    return pd.DataFrame(comparison_results)


def evaluate_and_compare_methods(data_path, splits_path, rigr_dir, native_dir, num_tasks, metrics, target_columns=None):
    """
    Convenience function to evaluate and compare RIGR vs Native methods.
    
    Args:
        data_path: Path to dataset CSV
        splits_path: Path to multiple_splits.json
        rigr_dir: Directory containing RIGR results
        native_dir: Directory containing Native/Chemprop results
        num_tasks: Number of target tasks
        metrics: List of metrics to compute
        target_columns: Target column names (optional)
    
    Returns:
        dict: Contains individual results and comparison
    """

    rigr_results = evaluate_results_multi_fold(
        data_path, splits_path, rigr_dir, num_tasks, metrics, target_columns
    )
    
    native_results = evaluate_results_multi_fold(
        data_path, splits_path, native_dir, num_tasks, metrics, target_columns
    )
    
    comparison = compare_methods_statistical(
        rigr_results, native_results, metrics, "RIGR", "Native"
    )
    
    return {
        'rigr': rigr_results,
        'native': native_results,
        'comparison': comparison
    }

## Barrier Cycloadd

In [27]:
# Dataset configuration
dataset_name = "barrier_cycloadd"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_cycloadd"
data_path = "/home/akshatz/bond_order_free/barriers_cycloadd/dataset/cycloadd_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["G_act"]

# Evaluate and compare methods
try:
    results = evaluate_and_compare_methods(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    with open(results_md_path, "w") as f:
        f.write("# Barrier Cycloadd\n\n")
        f.write("## Dataset: barrier_cycloadd\n\n")
        
        f.write("### RIGR Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['rigr']['summary'].to_markdown() + "\n\n")
        
        f.write("### Native Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['native']['summary'].to_markdown() + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## Barrier E2

In [28]:
# Dataset configuration
dataset_name = "barrier_e2"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_e2"
data_path = "/home/akshatz/bond_order_free/barriers_e2/dataset/e2_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["ea"]

# Evaluate and compare methods
try:
    results = evaluate_and_compare_methods(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    with open(results_md_path, "w") as f:
        f.write("# Barrier E2\n\n")
        f.write("## Dataset: barrier_e2\n\n")
        
        f.write("### RIGR Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['rigr']['summary'].to_markdown() + "\n\n")
        
        f.write("### Native Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['native']['summary'].to_markdown() + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## Barrier RDB7

In [29]:
# Dataset configuration
dataset_name = "barrier_rdb7"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_rdb7"
data_path = "/home/akshatz/bond_order_free/barriers_rdb7/dataset/rdb7_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["ea"]

# Evaluate and compare methods
try:
    results = evaluate_and_compare_methods(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    with open(results_md_path, "w") as f:
        f.write("# Barrier RDB7\n\n")
        f.write("## Dataset: barrier_rdb7\n\n")
        
        f.write("### RIGR Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['rigr']['summary'].to_markdown() + "\n\n")
        
        f.write("### Native Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['native']['summary'].to_markdown() + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## Barrier RGD1-CNHO

In [30]:
# Dataset configuration
dataset_name = "barrier_rgd1_cnho"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_rgd1_cnho"
data_path = "/home/akshatz/bond_order_free/barriers_rgd1/dataset/rgd1_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["ea"]

# Evaluate and compare methods
try:
    results = evaluate_and_compare_methods(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    with open(results_md_path, "w") as f:
        f.write("# Barrier RGD1-CNHO\n\n")
        f.write("## Dataset: barrier_rgd1_cnho\n\n")
        
        f.write("### RIGR Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['rigr']['summary'].to_markdown() + "\n\n")
        
        f.write("### Native Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['native']['summary'].to_markdown() + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Error during evaluation: operands could not be broadcast together with shapes (2,) (5,) 
Error during evaluation: operands could not be broadcast together with shapes (2,) (5,) 


Traceback (most recent call last):
  File "/tmp/ipykernel_2984653/3876677616.py", line 17, in <module>
    results = evaluate_and_compare_methods(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_2984653/2611510827.py", line 102, in evaluate_and_compare_methods
    comparison = compare_methods_statistical(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_2984653/2611510827.py", line 34, in compare_methods_statistical
    elif np.allclose(method1_values, method2_values):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/akshatz/anaconda3/envs/chemprop_benchmark_v2.0.3/lib/python3.11/site-packages/numpy/core/numeric.py", line 2241, in allclose
    res = all(isclose(a, b, rtol=rtol, atol=atol, equal_nan=equal_nan))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/akshatz/anaconda3/envs/chemprop_benchmark_v2.0.3/lib/python3.11/site-packages/numpy/core/numeric.py", line 2351, in isclose
    return wit

## Barrier SN2

In [31]:
# Dataset configuration
dataset_name = "barrier_sn2"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_sn2"
data_path = "/home/akshatz/bond_order_free/barriers_sn2/dataset/sn2_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["ea"]

# Evaluate and compare methods
try:
    results = evaluate_and_compare_methods(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    with open(results_md_path, "w") as f:
        f.write("# Barrier SN2\n\n")
        f.write("## Dataset: barrier_sn2\n\n")
        
        f.write("### RIGR Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['rigr']['summary'].to_markdown() + "\n\n")
        
        f.write("### Native Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['native']['summary'].to_markdown() + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## HIV

In [32]:
# Dataset configuration
dataset_name = "hiv"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/hiv"
data_path = "/home/akshatz/bond_order_free/hiv/dataset/hiv_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["prc-auc", "roc-auc", "ap"]
target_columns = ["HIV_active"]

# Evaluate and compare methods
try:
    results = evaluate_and_compare_methods(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    with open(results_md_path, "w") as f:
        f.write("# HIV\n\n")
        f.write("## Dataset: hiv\n\n")
        
        f.write("### RIGR Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['rigr']['summary'].to_markdown() + "\n\n")
        
        f.write("### Native Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['native']['summary'].to_markdown() + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## PCQM4MV2

In [33]:
# Dataset configuration
dataset_name = "pcqm4mv2"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/pcqm4mv2"
data_path = "/home/akshatz/bond_order_free/pcqm4mv2/dataset/pcqm4mv2_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["homolumogap"]

# Evaluate and compare methods
try:
    results = evaluate_and_compare_methods(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    with open(results_md_path, "w") as f:
        f.write("# PCQM4MV2\n\n")
        f.write("## Dataset: pcqm4mv2\n\n")
        
        f.write("### RIGR Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['rigr']['summary'].to_markdown() + "\n\n")
        
        f.write("### Native Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['native']['summary'].to_markdown() + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## QM9 Gap

In [34]:
# Dataset configuration
dataset_name = "qm9_gap"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/qm9/qm9_gap"
data_path = "/home/akshatz/bond_order_free/qm9/dataset/qm9_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["gap"]

# Evaluate and compare methods
try:
    results = evaluate_and_compare_methods(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    with open(results_md_path, "w") as f:
        f.write("# QM9 Gap\n\n")
        f.write("## Dataset: qm9_gap\n\n")
        
        f.write("### RIGR Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['rigr']['summary'].to_markdown() + "\n\n")
        
        f.write("### Native Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['native']['summary'].to_markdown() + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## QM9 U0

In [35]:
# Dataset configuration
dataset_name = "qm9_u0"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/qm9/qm9_u0"
data_path = "/home/akshatz/bond_order_free/qm9/dataset/qm9_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["u0_atom"]

# Evaluate and compare methods
try:
    results = evaluate_and_compare_methods(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    with open(results_md_path, "w") as f:
        f.write("# QM9 U0\n\n")
        f.write("## Dataset: qm9_u0\n\n")
        
        f.write("### RIGR Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['rigr']['summary'].to_markdown() + "\n\n")
        
        f.write("### Native Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['native']['summary'].to_markdown() + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## QM9 Multitask

In [36]:
# Dataset configuration
dataset_name = "qm9_multitask"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/qm9/qm9_multitask"
data_path = "/home/akshatz/bond_order_free/qm9/dataset/qm9_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 12
metrics = ["mae", "rmse", "r2"]
target_columns = None  # Auto-detect all 12 QM9 targets

# Evaluate and compare methods
try:
    results = evaluate_and_compare_methods(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    with open(results_md_path, "w") as f:
        f.write("# QM9 Multitask\n\n")
        f.write("## Dataset: qm9_multitask\n\n")
        
        f.write("### RIGR Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['rigr']['summary'].to_markdown() + "\n\n")
        
        f.write("### Native Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['native']['summary'].to_markdown() + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## UV/Vis

In [37]:
# Dataset configuration
dataset_name = "uv_vis"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/uv_vis"
data_path = "/home/akshatz/bond_order_free/multi_molecule/dataset/mult_mol_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["peakwavs_max"]

# Evaluate and compare methods
try:
    results = evaluate_and_compare_methods(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    with open(results_md_path, "w") as f:
        f.write("# UV/Vis\n\n")
        f.write("## Dataset: uv_vis\n\n")
        
        f.write("### RIGR Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['rigr']['summary'].to_markdown() + "\n\n")
        
        f.write("### Native Results (Mean ± Std across 5 folds)\n\n")
        f.write(results['native']['summary'].to_markdown() + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## PCBA Random

In [20]:
# Function for handling datasets with NaN targets (like PCBA)
def evaluate_results_multi_fold_with_nan(data_path, splits_path, result_dir, num_tasks, metrics, target_columns=None, num_folds=5):
    """
    Version of evaluate_results_multi_fold that handles NaN values in targets.
    Automatically detects the number of models in each fold.
    """
    df = pd.read_csv(data_path)
    with open(splits_path, "rb") as json_file:
        split_idxss = json.load(json_file)
    
    test_indices = parse_indices(split_idxss[0]["test"])
    test_df = df.iloc[test_indices]
    target_columns = test_df.keys()[-num_tasks:].tolist() if target_columns is None else target_columns
    
    metric_to_func = {metric: get_metric_func(metric) for metric in metrics}
    fold_results = defaultdict(list)
    
    for fold_idx in range(num_folds):
        fold_dir = os.path.join(result_dir, f"fold_{fold_idx}")
        
        # Automatically detect the number of models in this fold
        if not os.path.exists(fold_dir):
            print(f"Warning: Fold directory {fold_dir} does not exist")
            continue
            
        # Find all model directories in this fold
        model_dirs = [d for d in os.listdir(fold_dir) 
                     if os.path.isdir(os.path.join(fold_dir, d)) and d.startswith('model_')]
        
        if not model_dirs:
            print(f"Warning: No model directories found in fold {fold_idx}")
            continue
        
        # Sort model directories by model index
        model_dirs.sort(key=lambda x: int(x.split('_')[1]))
        
        # Collect predictions from all models in this fold
        df_pred_list = []
        for model_dir in model_dirs:
            model_file = os.path.join(fold_dir, model_dir, "test_predictions.csv")
            if os.path.exists(model_file):
                df_pred = pd.read_csv(model_file)[target_columns]
                df_pred_list.append(df_pred)
            else:
                print(f"Warning: Prediction file not found: {model_file}")
        
        if not df_pred_list:
            print(f"Warning: No predictions found for fold {fold_idx}")
            continue
            
        df_pred_fold = pd.concat(df_pred_list).groupby(level=0).mean()
        
        fold_metrics = defaultdict(list)
        for column in target_columns:
            for metric, metric_func in metric_to_func.items():
                preds = df_pred_fold[column].values
                targets = test_df[column].values
                
                # Vectorized NaN filtering (much faster)
                valid_mask = ~np.isnan(targets)
                if np.any(valid_mask):
                    targets_clean = targets[valid_mask]
                    preds_clean = preds[valid_mask]
                    try:
                        metric_value = metric_func(targets_clean, preds_clean)
                        fold_metrics[metric].append(metric_value)
                    except Exception as e:
                        print(f"Error calculating {metric} for fold {fold_idx}, column {column}: {e}")
                        fold_metrics[metric].append(np.nan)
                else:
                    fold_metrics[metric].append(np.nan)
        
        for metric in metrics:
            fold_results[metric].append(fold_metrics[metric])
    
    # Calculate summary statistics (same as before)
    summary_results = {}
    for metric in metrics:
        fold_metric_values = np.array(fold_results[metric])
        summary_results[f"{metric}_mean"] = np.nanmean(fold_metric_values, axis=0)
        summary_results[f"{metric}_std"] = np.nanstd(fold_metric_values, axis=0, ddof=1)
        summary_results[f"{metric}_folds"] = fold_metric_values
    
    summary_df = pd.DataFrame({
        metric + "_mean": summary_results[f"{metric}_mean"] for metric in metrics
    }, index=target_columns)
    
    for metric in metrics:
        summary_df[f"{metric}_std"] = summary_results[f"{metric}_std"]
    
    return {
        'summary': summary_df,
        'fold_results': dict(fold_results),
        'raw_results': summary_results
    }


def compare_methods_statistical_averaged(method1_results, method2_results, metrics, method1_name="Method1", method2_name="Method2", alpha=0.05):
    """
    Compare two methods using statistical tests on averaged metrics across all targets.
    This is appropriate for multi-target datasets like PCBA where we want one comparison per metric.
    
    Args:
        method1_results: Results from evaluate_results_multi_fold_with_nan for method 1
        method2_results: Results from evaluate_results_multi_fold_with_nan for method 2
        metrics: List of metrics to compare
        method1_name: Name of method 1 for display
        method2_name: Name of method 2 for display
    
    Returns:
        pd.DataFrame: Comparison results with p-values (one row per metric)
    """
    comparison_results = []
    
    for metric in metrics:
        
        method1_folds = method1_results['raw_results'][f"{metric}_folds"]  # Shape: (num_folds, num_targets)
        method2_folds = method2_results['raw_results'][f"{metric}_folds"]  # Shape: (num_folds, num_targets)
        
        # Average across all targets for each fold
        method1_fold_averages = np.nanmean(method1_folds, axis=1)  # Shape: (num_folds,)
        method2_fold_averages = np.nanmean(method2_folds, axis=1)  # Shape: (num_folds,)
        
        # Check for NaN values
        if np.any(np.isnan(method1_fold_averages)) or np.any(np.isnan(method2_fold_averages)):
            print(f"  Warning: NaN values found for averaged {metric}")
            t_stat, p_value = np.nan, np.nan
        elif len(method1_fold_averages) < 2 or len(method2_fold_averages) < 2:
            print(f"  Warning: Not enough samples for averaged {metric}")
            t_stat, p_value = np.nan, np.nan
        elif np.allclose(method1_fold_averages, method2_fold_averages):
            print(f"  Warning: Identical values for averaged {metric}")
            t_stat, p_value = 0.0, 1.0
        else:
            try:
                # Perform paired t-test on averaged values
                t_stat, p_value = ttest_rel(method1_fold_averages, method2_fold_averages)
            except Exception as e:
                print(f"  Error in t-test for averaged {metric}: {e}")
                t_stat, p_value = np.nan, np.nan
        
        # Calculate means and effect size
        method1_mean = np.nanmean(method1_fold_averages)
        method2_mean = np.nanmean(method2_fold_averages)
        method1_std = np.nanstd(method1_fold_averages, ddof=1)
        method2_std = np.nanstd(method2_fold_averages, ddof=1)
        
        # Cohen's d for effect size
        pooled_std = np.sqrt((method1_std**2 + method2_std**2) / 2)
        cohens_d = (method1_mean - method2_mean) / pooled_std if pooled_std > 0 and not np.isnan(pooled_std) else 0
        
        comparison_results.append({
            'metric': metric,
            f'{method1_name}_mean': method1_mean,
            f'{method1_name}_std': method1_std,
            f'{method2_name}_mean': method2_mean,
            f'{method2_name}_std': method2_std,
            't_statistic': t_stat,
            'p_value': p_value,
            'cohens_d': cohens_d,
            'significant': p_value < alpha if not np.isnan(p_value) else False
        })
    
    return pd.DataFrame(comparison_results)


def evaluate_and_compare_methods_with_nan(data_path, splits_path, rigr_dir, native_dir, num_tasks, metrics, target_columns=None):
    """
    Convenience function to evaluate and compare RIGR vs Native methods for datasets with NaN targets.
    Uses averaged statistical comparison for multi-target datasets.
    
    Args:
        data_path: Path to dataset CSV
        splits_path: Path to multiple_splits.json
        rigr_dir: Directory containing RIGR results
        native_dir: Directory containing Native/Chemprop results
        num_tasks: Number of target tasks
        metrics: List of metrics to compute
        target_columns: Target column names (optional)
    
    Returns:
        dict: Contains individual results and comparison
    """

    rigr_results = evaluate_results_multi_fold_with_nan(
        data_path, splits_path, rigr_dir, num_tasks, metrics, target_columns
    )
    
    native_results = evaluate_results_multi_fold_with_nan(
        data_path, splits_path, native_dir, num_tasks, metrics, target_columns
    )
    
    # Use averaged comparison for multi-target datasets
    comparison = compare_methods_statistical_averaged(
        rigr_results, native_results, metrics, "RIGR", "Native"
    )
    
    return {
        'rigr': rigr_results,
        'native': native_results,
        'comparison': comparison
    }

In [21]:
# Dataset configuration
dataset_name = "pcba_random"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/pcba/pcba_random"
data_path = "/home/akshatz/bond_order_free/pcba_random/dataset/pcba_random_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 128  # PCBA has 128 tasks
metrics = ["prc-auc", "roc-auc", "ap"]
target_columns = None  # Will auto-detect all 128 PCBA targets

# Evaluate and compare methods (using NaN-aware function for PCBA)
try:
    results = evaluate_and_compare_methods_with_nan(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    
    with open(results_md_path, "w") as f:
        f.write("# PCBA Random\n\n")
        f.write("## Dataset: pcba_random\n\n")
        
        f.write("### RIGR Results (Averaged Across All Targets)\n\n")
        f.write(results['comparison'][['metric', 'RIGR_mean', 'RIGR_std']].to_markdown(index=False) + "\n\n")
        
        f.write("### Native Results (Averaged Across All Targets)\n\n")
        f.write(results['comparison'][['metric', 'Native_mean', 'Native_std']].to_markdown(index=False) + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
        
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## PCBA Random NaN

In [ ]:
# Dataset configuration
dataset_name = "pcba_random_nan"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/pcba/pcba_random_nan"
data_path = "/home/akshatz/bond_order_free/pcba_random_nan/dataset/pcba_random_nan_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 128  # PCBA has 128 tasks
metrics = ["prc-auc", "roc-auc", "ap"]
target_columns = None  # Will auto-detect all 128 PCBA targets

# Evaluate and compare methods (using NaN-aware function for PCBA)
try:
    results = evaluate_and_compare_methods_with_nan(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    
    with open(results_md_path, "w") as f:
        f.write("# PCBA Random NaN\n\n")
        f.write("## Dataset: pcba_random_nan\n\n")
        
        f.write("### RIGR Results (Averaged Across All Targets)\n\n")
        f.write(results['comparison'][['metric', 'RIGR_mean', 'RIGR_std']].to_markdown(index=False) + "\n\n")
        
        f.write("### Native Results (Averaged Across All Targets)\n\n")
        f.write(results['comparison'][['metric', 'Native_mean', 'Native_std']].to_markdown(index=False) + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
        
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## PCBA Scaffold

In [ ]:
# Dataset configuration
dataset_name = "pcba_scaffold"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/pcba/pcba_scaffold"
data_path = "/home/akshatz/bond_order_free/pcba_scaffold/dataset/pcba_scaffold_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 127  # PCBA Scaffold has 127 tasks
metrics = ["prc-auc", "roc-auc", "ap"]
target_columns = None  # Will auto-detect all PCBA targets

# Evaluate and compare methods (using NaN-aware function for PCBA)
try:
    results = evaluate_and_compare_methods_with_nan(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns
    )
    
    with open(results_md_path, "w") as f:
        f.write("# PCBA Scaffold\n\n")
        f.write("## Dataset: pcba_scaffold\n\n")
        
        f.write("### RIGR Results (Averaged Across All Targets)\n\n")
        f.write(results['comparison'][['metric', 'RIGR_mean', 'RIGR_std']].to_markdown(index=False) + "\n\n")
        
        f.write("### Native Results (Averaged Across All Targets)\n\n")
        f.write(results['comparison'][['metric', 'Native_mean', 'Native_std']].to_markdown(index=False) + "\n\n")
        
        f.write("### Statistical Comparison (RIGR vs Native)\n\n")
        f.write(results['comparison'].to_markdown() + "\n\n")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

# SAMPL

In [47]:
def evaluate_sampl(test_no, result_dir, metrics):
    targets = None
    df_pred_list = []
    files = glob.glob(os.path.join(result_dir, '**', f"pred_SAMPL{test_no}.csv"), recursive=True)
    assert len(files) == 1, f"There should be 1 file; {len(files)} found"
    for file in files:
        df = pd.read_csv(file)
        
        if targets is None:
            targets = df["logP mean"].tolist() if test_no != 9 else df["new_logPexp_reviewed"].tolist()
            
        df_pred = df["pred_0"]
        df_pred_list.append(df_pred)
    preds = pd.concat(df_pred_list).groupby(level=0).mean().tolist()
    
    metric_to_func = {metric: get_metric_func(metric) for metric in metrics}

    results = defaultdict(list)
    for metric, metric_func in metric_to_func.items():
        results[metric].append(metric_func(targets, preds))
    results = dict(results)

    results_df = pd.DataFrame(results, index=[f"logP - SAMPL{test_no}"])
    return results_df

In [48]:
metrics = metrics = ["mae", "rmse", "r2"]
evaluate_sampl(6, "/home/akshatz/bond_order_free/logp/run1_bof/results_sampl_production", metrics)

,mae,rmse,r2
logP - SAMPL6,0.323174,0.391207,0.655518


In [49]:
metrics = metrics = ["mae", "rmse", "r2"]
evaluate_sampl(7, "/home/akshatz/bond_order_free/logp/run1_bof/results_sampl_production", metrics)

,mae,rmse,r2
logP - SAMPL7,0.358151,0.501945,0.428742


In [50]:
metrics = metrics = ["mae", "rmse", "r2"]
evaluate_sampl(9, "/home/akshatz/bond_order_free/logp/run1_bof/results_sampl_production", metrics)

,mae,rmse,r2
logP - SAMPL9,0.968365,1.122327,0.749068
